# Colab Runner — grille scope x mécanisme (Full/Selective FT vs LoRA)

**Notebook principal de l'expérimentation.** Il ne contient aucune logique métier :
il récupère la dernière version du code (`git clone`/`pull`), puis pilote
`src/forensic_fr` via des paramètres modifiables ci-dessous (cellule *Paramètres*).

Marche à suivre :
1. `Runtime > Change runtime type` → GPU
2. Exécuter les cellules **Récupération du code** et **Environnement** une fois
3. Ajuster la cellule **Paramètres** selon le run voulu, puis l'exécuter
4. Exécuter **Lancement**

Pour relancer un autre volet (ex. passer d'un run unique à la grille complète, ou
activer le contrôle sans ancrage), il suffit de rééditer la cellule *Paramètres* et
de ré-exécuter à partir de là — pas besoin de toucher au code source.

## 1. Récupération du code

`git clone` si le dépôt n'existe pas encore dans la session, `git pull` sinon — donc
toujours la dernière version poussée sur GitHub, y compris en relançant cette cellule
dans une session déjà démarrée.

Pour un dépôt privé : ajouter un secret Colab nommé `GITHUB_TOKEN` (icône clé dans la
barre latérale gauche) plutôt que de coller un token en clair dans `REPO_URL` — ce
notebook est versionné sur GitHub, tout ce qui est écrit ici est public.

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/MarouaneAyech/foresynch_b2.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
REPO_DIR = "/content/foresynch_b2"  #@param {type:"string"}

try:
    from google.colab import userdata
    _token = userdata.get("GITHUB_TOKEN")
except Exception:
    _token = None

clone_url = REPO_URL
if _token and REPO_URL.startswith("https://"):
    clone_url = REPO_URL.replace("https://", f"https://{_token}@")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"Depot deja present dans {REPO_DIR} -> fetch + checkout + pull ({BRANCH})")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
else:
    print(f"Clonage de {REPO_URL} (branche {BRANCH}) dans {REPO_DIR}")
    subprocess.run(["git", "clone", "-b", BRANCH, clone_url, REPO_DIR], check=True)

import sys
if os.path.join(REPO_DIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))
print("src ajoute au PYTHONPATH :", os.path.join(REPO_DIR, "src"))

## 2. Environnement

Montage de Drive (données SCface, caches, checkpoints, historiques — voir la table
des dépendances du notebook d'origine) et installation des dépendances.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

FORENSIC_FR_ROOT = "/content/drive/MyDrive/research/forensic_fr"  #@param {type:"string"}
os.environ["FORENSIC_FR_ROOT"] = FORENSIC_FR_ROOT

!pip install -q -r {REPO_DIR}/src/requirements.txt
!pip install -q insightface onnxruntime-gpu

from forensic_fr import config as cfg
cfg.bootstrap_configs(REPO_DIR)  # copie configs/phase1_finetune/*.json du depot vers Drive si absent

## 3. Paramètres de l'expérimentation

C'est la seule cellule à modifier pour changer de volet. `RUN_MODE` bascule entre un
run unique (debug, un contrôle ciblé) et la grille complète ou un sous-ensemble.

Exemples de sous-ensembles utiles (`GRID_ONLY`) :
- `ft_34,full_ft,lora_34` → les 3 configs prioritaires de la Phase 1 (analyse spectrale)
- `ft_34` avec `ANCHOR` décoché → contrôle FT-sans-ancrage (Phase 4.3.2.C du plan)
- vide → grille complète (8 configs x seeds = jusqu'à 24 runs)

In [ ]:
#@markdown ### Volet à exécuter
RUN_MODE = "single"  #@param ["single", "grid"]

#@markdown ### Paramètres communs
EXPERIMENT_ID = "E5_LoRA"  #@param {type:"string"}
N_EPOCHS = 20  #@param {type:"integer"}
ANCHOR = True  #@param {type:"boolean"}

#@markdown ### Si RUN_MODE = "single"
MODE = "lora_34"  #@param ["full_ft", "ft_34", "ft_4", "full_lora", "lora_34", "lora_4", "hybrid"]
SEED = 42  #@param {type:"integer"}
LORA_R = 8  #@param [8, 16]
LORA_ALPHA = 16  #@param {type:"integer"}

#@markdown ### Si RUN_MODE = "grid" — laisser GRID_ONLY vide = toute la grille
GRID_ONLY = ""  #@param {type:"string"}
GRID_SEEDS = "7,42,123"  #@param {type:"string"}

## 4. Lancement

In [ ]:
from forensic_fr.training import RunConfig, build_plan, run_training

if RUN_MODE == "single":
    rc = RunConfig(
        mode=MODE, seed=SEED, lora_r=LORA_R, lora_alpha=LORA_ALPHA,
        anchor=ANCHOR, n_epochs=N_EPOCHS, experiment_id=EXPERIMENT_ID,
    )
    result = run_training(rc)
    print("Checkpoint :", result["checkpoint_path"])
    print("Historique :", result["history_path"])

elif RUN_MODE == "grid":
    only = [m.strip() for m in GRID_ONLY.split(",") if m.strip()] or None
    seeds = [int(s.strip()) for s in GRID_SEEDS.split(",") if s.strip()]
    plan = build_plan(only=only, seeds=seeds, anchor=ANCHOR,
                       n_epochs=N_EPOCHS, experiment_id=EXPERIMENT_ID)

    print(f"{len(plan)} run(s) planifies :")
    for rc in plan:
        print(f"  - {rc.mode} r={rc.lora_r} seed={rc.seed} anchor={rc.anchor}")

    for i, rc in enumerate(plan, 1):
        print(f"\n[{i}/{len(plan)}] === {rc.mode} seed={rc.seed} ===")
        run_training(rc)

else:
    raise ValueError(f"RUN_MODE inconnu : {RUN_MODE!r}")

## 5. Résultats produits

Checkpoints (`.pt`) et historiques (`exp1_{mode}.json`) du run (Drive, persistant
entre sessions).

In [ ]:
from forensic_fr import config as cfg

run_dir = cfg.run_dir(EXPERIMENT_ID)

print("Checkpoints :")
for p in sorted((run_dir / "checkpoints").glob("*.pt")):
    print(" -", p)

print("\nHistoriques :")
for p in sorted(run_dir.glob("exp1_*.json")):
    print(" -", p)